In [1]:
import os

# policy
from huggingface_hub import snapshot_download
from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.pretrained import PreTrainedPolicy
from lerobot.policies.act.modeling_act import ACTPolicy

# record utils
from lerobot.record import record, RecordConfig, DatasetRecordConfig

# torch
from torch import cuda

# utils
from dotenv import load_dotenv

# my code
from src.robot_config import robot_config
from src.paths import REPO_ROOT, HF_NAME, POLICIES_DIR, EVAL_DIR
from src.robot_const import TABLE_START_POSE_OPEN
from src.utils import check_resume

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

/home/jonathan/miniconda3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Set Params:

In [ ]:
PUSH_TO_HUB       = False
SAVE_TO_DATASET   = True
REPO_NAME         = 'so101_leg_pick_and_place'
EXPERIMENT_NAME   = '50_episodes_v1'
POLICY_CHECKPOINT = '100000'
POLICY_TYPE       = 'act'
NUM_EPISODES      = 5
FPS               = 30
EPISODE_TIME_SEC  = 60
RESET_TIME_SEC    = 5
EVAL_TYPE         = 'real_v0'

Log to HF if pushing:

In [3]:
if PUSH_TO_HUB:
    os.system(f"huggingface-cli login --token {os.getenv('HUGGINGFACE_TOKEN')}")

Initialize Policy:

In [4]:
# Resolve path or HF id
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / "checkpoints" / POLICY_CHECKPOINT / "pretrained_model"

if not policy_path.exists():
    # load from Hugging Face Hub
    policy_id = f"{HF_NAME}/{POLICY_TYPE}-{REPO_NAME}-{EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = policy_id,
    revision               = "main",
    local_dir              = str(policy_path),
    local_dir_use_symlinks = False
    )

policy_config = PreTrainedConfig.from_pretrained(policy_path)
policy_config.pretrained_path = policy_path

Build Dataset:

In [ ]:
dataset_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"
resume = check_resume(dataset_path)

dscfg = DatasetRecordConfig(
    repo_id                             = f"{HF_NAME}/eval_{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    single_task                         = f"eval dataset for {REPO_NAME} with policy {EXPERIMENT_NAME}, mode = {EVAL_TYPE}",
    root                                = dataset_path.__str__(),
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 0,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = None,
    policy       = policy_config,
    display_data = True,
    play_sounds  = True,
    resume       = resume
)

In [ ]:
record(rc, reset_pose = TABLE_START_POSE_OPEN, give_feedback = True)

Loading weights from local directory


INFO 2025-09-27 19:23:40 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-Sonix_Technology_Co.__Ltd._USB2.0_CAM1_USB2.0_CAM1-video-index0) connected.
INFO 2025-09-27 19:23:42 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-046d_Logitech_BRIO_8F54E371-video-index0) connected.
INFO 2025-09-27 19:23:42 follower.py:104 so_101_follower SO101Follower connected.
INFO 2025-09-27 19:23:43 ls/utils.py:227 Recording episode 0
[2025-09-27T16:23:43Z INFO  winit::platform_impl::linux::x11::window] Guessed window scale factor: 1
[2025-09-27T16:23:43Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-09-27T16:23:44Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-09-27T16:23:44Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-09-27T16:23:44Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-09-27T16:23:44Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-09-27T16:23:44Z WARN  wgpu_hal

Escape key pressed. Stopping data recording...


OSError: [Errno 39] Directory not empty: '/home/jonathan/Documents/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_leg_pick_and_place/50_episodes_v1-real_v0/images/observation.images.wrist_cam/episode-000000'

Error writing image /home/jonathan/Documents/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_leg_pick_and_place/50_episodes_v1-real_v0/images/observation.images.top_cam/episode-000000/frame-001448.png: [Errno 2] No such file or directory: '/home/jonathan/Documents/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_leg_pick_and_place/50_episodes_v1-real_v0/images/observation.images.top_cam/episode-000000/frame-001448.png'
Error writing image /home/jonathan/Documents/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_leg_pick_and_place/50_episodes_v1-real_v0/images/observation.images.wrist_cam/episode-000000/frame-001449.png: [Errno 2] No such file or directory: '/home/jonathan/Documents/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_leg_pick_and_place/50_episodes_v1-real_v0/images/observation.images.wrist_cam/episode-000000/frame-001449.png'
Error writing image /home/jonathan/Documents/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_leg_pick_and_place/50_episodes_v1-real_v0/images/observation.ima